In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.optimize import curve_fit
from scipy import special
import pandas as pd
import math
import sys
import os
import ROOT

dir = "/Users/alexanderantonakis/Desktop/Software/AFrameAnalysis/Macros/"

sys.path.append("../Utils")
sys.path.append("../Configs")
sys.path.append("../DAQ")
sys.path.append("../Mappings")
sys.path.append("../BiasFiles")
sys.path.append("../SiPM_HISTS")

from frame_to_crt_fcl import *

def linear_fit(x, m, b):
    return m*x + b

def linear_inv(m, b, goal):
    x = (goal - b)/m
    if 40 < x < 90:
        return x
    else:
        return -1

def eff_func(x, a, b, c):
        return a*special.erf(x-c, out=None) + b

print("made some eff functions")

In [ ]:
x = np.linspace(66, 68, 1000)

y1 = eff_func(x, 1, 0, 66.1)
y2 = eff_func(x, 0.9, 0, 66)
plt.plot(x, y1, c="b")
plt.plot(x, y2, c="r")
plt.plot(x, y2/y1, c="g")
plt.ylim(0.4, 1.25)
plt.show()

In [ ]:
all_sipm_hists = [f for f in os.listdir("../SiPM_HISTS/") if os.path.isfile("../SiPM_HISTS/"+f)]


print("All SiPM Hists available")
print(all_sipm_hists)

In [ ]:
from run_to_voltage import*
volt_df = pd.DataFrame(run_to_volt, columns = ["Frame", "Run", "V"])
volt_df[:4]

In [ ]:
daq_df = pd.DataFrame(frame_map, columns = ["Frame", "Frame_FEB", "DAQ_FEB", "Wall"])
daq_df[:4]

In [ ]:
frame = 7

frame7_febs = list(daq_df.query("Frame == 7")["Frame_FEB"].values)
print(frame7_febs)

In [ ]:
runs = volt_df.query("Frame == "+str(frame))["Run"].values
voltages = volt_df.query("Frame == "+str(frame))["V"].values
sorted_voltages, sorted_runs = zip(*sorted(zip(voltages, runs)))
# Convert them back to lists (zip returns tuples)
sorted_voltages = list(sorted_voltages)
sorted_runs = list(sorted_runs)
print("Runs", sorted_runs)
print("volt", sorted_voltages)

f = ROOT.TFile.Open("../SiPM_HISTS/sipm_hists_frame"+str(frame)+".root", "READ")

In [ ]:
frame_feb = 197

fractions = [[] for num in range(len(sorted_runs))]

count = 0
for run in runs:
    odd_name = str(run)+"_"+str(frame_feb)+"_odd"
    even_name = str(run)+"_"+str(frame_feb)+"_even"
    h_odd = f.Get(odd_name)
    h_even = f.Get(even_name)
    h_ratio = h_even.Clone("h_ratio_"+str(frame_feb))
    h_ratio.Divide(h_odd)
    for num in range(16):
        fractions[count].append(h_ratio.GetBinContent(num+1))
            
    count += 1

In [ ]:
y_test = [fractions[num][0] for num in range(len(sorted_voltages))]
plt.scatter(sorted_voltages, y_test)
plt.show()

In [ ]:
def fit_func(x, a, b1, b2):
    return a*(special.erf(x-b1, out=None)/special.erf(x-b2, out=None))


popt, pcov = curve_fit(fit_func, sorted_voltages, y_test, p0=[1.0, 130, 150])

x_fit = np.linspace(min(sorted_voltages), max(sorted_voltages), 1000)
y_fit = fit_func(x_fit, popt[0], popt[1], popt[2])

plt.scatter(sorted_voltages, y_test)
plt.plot(x_fit, y_fit, c="r")
plt.show()


In [ ]:
x = np.linspace(65, 68, 1000)

y = eff_func(x, 4, -3, 65.5)

plt.plot(x, y, c="b")

#plt.ylim(0.4, 1.25)
plt.show()

In [ ]:
x = np.linspace(-2, 2, 1000)
y = eff_func(x, 0.5, 0.5, 0)

plt.plot(x, y)
plt.show()